### RAG pipeline - Data ingestion to Vector Db Pipeline

In [25]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [26]:
### read all the PDF  inside directory 

def process_all_pdfs(pdf_directory):
    """ Process all pdf files inside directory """
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF to process")
    for pdf_file in pdf_files:
        print(f"\nprocessing : {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']="pdf"
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"error is {e}")
    print(f"Total documents processed are {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")

found 3 PDF to process

processing : Designing_Data_Intensive_Applications_TH.pdf
loaded 613 pages

processing : github-foundations-exam-study-guide.pdf
loaded 8 pages

processing : shivam_tripathi_v2.pdf
loaded 1 pages
Total documents processed are 622


In [27]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 15.3.1 (Build 24D70) Quartz PDFContext, AppendMode 1.1', 'creator': '', 'creationdate': "D:20191214160412Z00'00'", 'keywords': '', 'author': '', 'title': '', 'moddate': "D:20250812065820Z00'00'", 'subject': '', 'source': '../data/pdf/Designing_Data_Intensive_Applications_TH.pdf', 'total_pages': 613, 'page': 0, 'page_label': 'Cover', 'source_file': 'Designing_Data_Intensive_Applications_TH.pdf', 'file_type': 'pdf'}, page_content='Martin Kleppmann\nDesigning \nData-Intensive \nApplications\nTHE BIG IDEAS BEHIND RELIABLE, SCALABLE,  \nAND MAINTAINABLE SYSTEMS'),
 Document(metadata={'producer': 'macOS Version 15.3.1 (Build 24D70) Quartz PDFContext, AppendMode 1.1', 'creator': '', 'creationdate': "D:20191214160412Z00'00'", 'keywords': '', 'author': '', 'title': '', 'moddate': "D:20250812065820Z00'00'", 'subject': '', 'source': '../data/pdf/Designing_Data_Intensive_Applications_TH.pdf', 'total_pages': 613, 'page': 1, 'page_label': 'BackCover', '

In [28]:
### TExt Splitting into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ split the documents into chunks for better RAG Performance """
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("example chunk ")
        print(f"content :{split_docs[0].page_content[:200]}...")
        print(f"metadata : {split_docs[0].metadata}")
    return split_docs   


In [29]:
chunks=split_documents(all_pdf_documents)

split 622 documents into 1996 chunks
example chunk 
content :Martin Kleppmann
Designing 
Data-Intensive 
Applications
THE BIG IDEAS BEHIND RELIABLE, SCALABLE,  
AND MAINTAINABLE SYSTEMS...
metadata : {'producer': 'macOS Version 15.3.1 (Build 24D70) Quartz PDFContext, AppendMode 1.1', 'creator': '', 'creationdate': "D:20191214160412Z00'00'", 'keywords': '', 'author': '', 'title': '', 'moddate': "D:20250812065820Z00'00'", 'subject': '', 'source': '../data/pdf/Designing_Data_Intensive_Applications_TH.pdf', 'total_pages': 613, 'page': 0, 'page_label': 'Cover', 'source_file': 'Designing_Data_Intensive_Applications_TH.pdf', 'file_type': 'pdf'}


### Embedding and Vectore store DB 


In [30]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List , Dict,Any, Tuple 
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """ Load sentence transformer model """
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully , Model dimensions are {self.model.get_embedding_dimension()}")
        except Exception as e :
            print(f"exception occured :{e}")
            raise e
    def generate_embeddings(self,texts:List[str])-> np.ndarray:
        """
        generate Embeddings for list of text 

        ARGS:
            texts: List of text strings to embed 
        return :
        numpy array of embedding of shape (len(texts),embedding_dimensions)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate embeddings for {len(texts)}...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"GEnerated embedding with shape {embeddings.shape}")
        return embeddings
    
emnbedding_manager=EmbeddingManager()
emnbedding_manager



Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6810.65it/s]


Model loaded successfully , Model dimensions are 384


### Vector Store


In [32]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        initialise vector store 
        
        Args:
            collection name =name of the chromadb collection 
            persist_directory = Directory to persist vector store  
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()
        
    def _initialise_store(self):
        """initialise chroma db client and collection"""
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection= self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for rag"}
            )
            print(f"Vector store initialised , collection f{self.collection_name}")
            print(f"existing documents in collection {self.collection.count()}")

        except Exception as e:
            print(f"error occureed  while initialising vector store,{e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        add documents and there embeedding to vector store 
        ARGs:
            documents: list of langchain documents
            embedding : corresponding embedding for the documents

        """
        if len(documents)!= len(embeddings):
            raise ValueError("number of documents must match number of embeddings")
        print(f"adding {len(documents)} documents to the vector store ")
        
        # prepare for chromaDB 
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # generate unique id 
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare  metadata 
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #documents content

            documents_text.append(doc.page_content)

            # Embeddings 
            embeddings_list.append(embedding.tolist())
        
        # add to collection 
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
                )
            print(f"successfully added {len(documents)} documetns to vector store ")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print("error inserting document sint vector store ")
            raise
vectorstore=VectorStore()
vectorstore




Vector store initialised , collection fpdf_documents
existing documents in collection 80


In [33]:
chunks

[Document(metadata={'producer': 'macOS Version 15.3.1 (Build 24D70) Quartz PDFContext, AppendMode 1.1', 'creator': '', 'creationdate': "D:20191214160412Z00'00'", 'keywords': '', 'author': '', 'title': '', 'moddate': "D:20250812065820Z00'00'", 'subject': '', 'source': '../data/pdf/Designing_Data_Intensive_Applications_TH.pdf', 'total_pages': 613, 'page': 0, 'page_label': 'Cover', 'source_file': 'Designing_Data_Intensive_Applications_TH.pdf', 'file_type': 'pdf'}, page_content='Martin Kleppmann\nDesigning \nData-Intensive \nApplications\nTHE BIG IDEAS BEHIND RELIABLE, SCALABLE,  \nAND MAINTAINABLE SYSTEMS'),
 Document(metadata={'producer': 'macOS Version 15.3.1 (Build 24D70) Quartz PDFContext, AppendMode 1.1', 'creator': '', 'creationdate': "D:20191214160412Z00'00'", 'keywords': '', 'author': '', 'title': '', 'moddate': "D:20250812065820Z00'00'", 'subject': '', 'source': '../data/pdf/Designing_Data_Intensive_Applications_TH.pdf', 'total_pages': 613, 'page': 2, 'page_label': 'i', 'source_f

In [34]:
### convert the text to embeddings 

texts=[doc.page_content for doc in chunks]
### generate the embeddings 
embeddings=emnbedding_manager.generate_embeddings(texts)

### store in the vector database 
vectorstore.add_documents(chunks,embeddings)



Generate embeddings for 1996...


Batches: 100%|██████████| 63/63 [00:26<00:00,  2.42it/s]


GEnerated embedding with shape (1996, 384)
adding 1996 documents to the vector store 
successfully added 1996 documetns to vector store 
Total documents in collection: 2076


### Retrival pipeline from vector store

In [35]:
class RagRetriver:
    """
    handles query based retrival from vector store 
    """
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager): 
        """
        Initialise  the retriver 

        args :
            vector_store: vector store containing the document embeddings
            embedding manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevent documents for the query 
        args :

            query : the search query 
            top_k: NUmber of top results to show 
            score_threshold: Minimum similarity score threshold 
        
        returns :
            List of dictionaries containing retrived documents and metadata 
        
        """
        print(f"retrieving  document for the query {query}")
        print(f"top_k={top_k}, and score_threshold is = {score_threshold}")
        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas= results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i , (doc_id, document,metadata, distance) in enumerate(zip(ids,documents, metadatas,distances)):
                    similarity_score=1-distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"we retrieved {len(retrieved_docs)} documents after filtering ")

            else:
                print(" no documents found ")
            return retrieved_docs
        except Exception as e :
            print(f"found error as {e}")
            return []

rag_retriever=RagRetriver(vectorstore,emnbedding_manager)
    


In [36]:
rag_retriever

In [ ]:
rag_retriever.retrieve("what are transactions ",top_k=2)

retrieving  document for the query what are transactions 
top_k=1, and score_threshold is = 0.0
Generate embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]

GEnerated embedding with shape (1, 384)
we retrieved 1 documents after filtering 


[{'id': 'doc_4f1cea2c_705',
  'content': 'For decades, transactions have been the mechanism of choice for simplifying these\nissues. A transaction is a way for an application to group several reads and writes\ntogether into a logical unit. Conceptually, all the reads and writes in a transaction are\nexecuted as one operation: either the entire transaction succeeds ( commit) or it fails\n(abort, rollback). If it fails, the application can safely retry. With transactions, error\nhandling becomes much simpler for an application, because it doesn’t need to worry\nabout partial failure—i.e., the case where some operations succeed and some fail (for\nwhatever reason).\nIf you have spent years working with transactions, they may seem obvious, but we\nshouldn’t take them for granted. Transactions are not a law of nature; they were cre‐\nated with a purpose, namely to simplify the programming model  for applications\naccessing a database. By using transactions, the application is free to ignore

### Integration Vector DB  Context pipeline with LLM Output 

In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key=os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    print("no key avialable")

llm = ChatOpenAI(
    model="gpt-5-nano",
    # stream_usage=True,
    temperature=0.1,
    max_tokens=500,

    # timeout=None,
    # reasoning_effort="low",
    # max_retries=2,
    # api_key="...",  # If you prefer to pass api key in directly
    # base_url="...",
    # organization="...",
    # other params...
)

### simple rag function retrieve context and generate response 
def rag_simple(query:str, retriever:RagRetriver, llm:ChatOpenAI,top_k=1):
    # retrieve the context 
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    context=""
    if not context:
        return "NO relevent context found to answer the question."
    ### generate answer using openai llm 
    prompt=f"""
    
            use the following context to aswer the question 
            context:
            {context}

            question = {query}

            answer:?
            
            
            """
    print(prompt)
    response=llm.invoke([prompt.format(context=context,query=query)])
    print(response)
    return response.content


In [66]:
answer=rag_simple("what are transactions  ",rag_retriever,llm)
print(answer)

retrieving  document for the query what are transactions  
top_k=1, and score_threshold is = 0.0
Generate embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]


GEnerated embedding with shape (1, 384)
we retrieved 1 documents after filtering 


            use the following context to aswer the question 
            context:
            For decades, transactions have been the mechanism of choice for simplifying these
issues. A transaction is a way for an application to group several reads and writes
together into a logical unit. Conceptually, all the reads and writes in a transaction are
executed as one operation: either the entire transaction succeeds ( commit) or it fails
(abort, rollback). If it fails, the application can safely retry. With transactions, error
handling becomes much simpler for an application, because it doesn’t need to worry
about partial failure—i.e., the case where some operations succeed and some fail (for
whatever reason).
If you have spent years working with transactions, they may seem obvious, but we
shouldn’t take them for granted. Transactions are not a law of nature; they were cre‐
ated with a purpose, namely to si